<a href="https://colab.research.google.com/github/dhag/colab_test/blob/main/AI_04_B%E5%AE%9F%E7%BF%92B_%E3%83%86%E3%82%A3%E3%83%BC%E3%83%81%E3%83%A3%E3%83%96%E3%83%AB%E3%83%9E%E3%82%B7%E3%83%B3%E3%81%A7%E4%BD%9C%E3%81%A3%E3%81%9F%E3%83%A2%E3%83%87%E3%83%AB%E3%81%A7%E3%82%A4%E3%83%B3%E3%82%BF%E3%83%A9%E3%82%AF%E3%83%86%E3%82%A3%E3%83%96%E3%81%AA%E7%94%BB%E5%83%8F%E8%AA%8D%E8%AD%98_%E3%82%B9%E3%83%88%E3%83%AA%E3%83%BC%E3%83%9F%E3%83%B3%E3%82%B0%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Teachable Machineモデルでインタラクティブな画像認識

このノートブックは、Google Colab上でTeachable Machineから出力した `.h5` モデルを読み込み、Webカメラ画像をGradioで認識します。

## 初回のみ
最初のセルを実行した後、表示された指示に従って**ランタイムを再起動（[ランタイム][セッションの再起動]）**してください。再起動後、上から順番に実行します。


In [1]:
!pip install -q tf-keras gradio


## ランタイム再起動後

モデルファイル（`.h5`）とラベルファイル（`.txt`）をアップロードします。


In [1]:
# ==========================================
# ライブラリ
# ==========================================
import os
import json
import shutil
import tempfile
import re

import cv2
import numpy as np
from PIL import Image

import tensorflow as tf
import tf_keras
import gradio as gr
import h5py

from google.colab import files


print("TensorFlow :", tf.__version__)
print("tf_keras  :", tf_keras.__version__)
print("Gradio    :", gr.__version__)


# ==========================================
# H5モデルの互換性修正
# ==========================================
def _remove_groups_in_config(obj):
    """
    古いTeachable Machine H5と新しいKerasの互換性対策。
    DepthwiseConv2Dのconfigに入っているgroupsを削除する。
    """
    if isinstance(obj, dict):
        if obj.get("class_name") == "DepthwiseConv2D":
            config = obj.get("config", {})
            config.pop("groups", None)

        for key in list(obj.keys()):
            obj[key] = _remove_groups_in_config(obj[key])

        return obj

    if isinstance(obj, list):
        return [_remove_groups_in_config(x) for x in obj]

    return obj


def make_compatible_h5(src_path):
    """
    元のH5ファイルを変更せず、互換性修正版の一時ファイルを作る。
    """
    fd, dst_path = tempfile.mkstemp(suffix=".h5")
    os.close(fd)
    shutil.copy2(src_path, dst_path)

    with h5py.File(dst_path, "r+") as f:
        if "model_config" not in f.attrs:
            raise RuntimeError("H5ファイルに model_config がありません。")

        model_config = f.attrs["model_config"]

        if isinstance(model_config, bytes):
            model_config = model_config.decode("utf-8")

        config = json.loads(model_config)
        config = _remove_groups_in_config(config)

        # 元の属性と同じJSON文字列として保存
        f.attrs.modify(
            "model_config",
            json.dumps(config).encode("utf-8")
        )

    return dst_path


def safe_load_model(model_path):
    """
    Teachable Machineの古いH5をtf_kerasで読み込む。

    1. まず元ファイルをそのまま読み込む
    2. 失敗したら一時コピーからgroupsを除去して再試行
    3. 元のH5ファイルは変更しない
    """

    # ---- 1. そのまま読み込み ----
    try:
        model = tf_keras.models.load_model(
            model_path,
            compile=False
        )

        print("✓ tf_kerasでモデルを読み込みました")
        return model

    except Exception as e:
        print("通常の読み込みに失敗しました。")
        print(f"  {type(e).__name__}: {e}")

    # ---- 2. 一時コピーを修正して読み込み ----
    fixed_path = None

    try:
        fixed_path = make_compatible_h5(model_path)

        model = tf_keras.models.load_model(
            fixed_path,
            compile=False
        )

        print("✓ 互換性修正版モデルを読み込みました")
        print("✓ 元のH5ファイルは変更していません")

        return model

    except Exception as e:
        raise RuntimeError(
            "Teachable MachineのH5モデルを読み込めませんでした。\n"
            f"エラー: {type(e).__name__}: {e}"
        ) from e

    finally:
        # 一時ファイルを削除
        if fixed_path is not None and os.path.exists(fixed_path):
            try:
                os.remove(fixed_path)
            except Exception:
                pass


# ==========================================
# モデル・ラベルのアップロード
# ==========================================
print("Teachable Machineでエクスポートしたモデル（.h5）をアップロードしてください。")
uploaded_model = files.upload()

if not uploaded_model:
    raise RuntimeError("モデルファイルがアップロードされませんでした。")

model_path = next(iter(uploaded_model.keys()))


print("\nラベルファイル（.txt）をアップロードしてください。")
uploaded_labels = files.upload()

if not uploaded_labels:
    raise RuntimeError("ラベルファイルがアップロードされませんでした。")

label_path = next(iter(uploaded_labels.keys()))


# ==========================================
# ラベル読み込み
# ==========================================
def load_labels(path):
    """
    Teachable Machineのlabels.txtを読み込む。

    例:
        0 classA
        1 classB

    の場合はclassA/classBだけをラベルとして使用する。
    """
    labels = []

    with open(path, "r", encoding="utf-8-sig") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            # "0 class_name" / "0: class_name" などに対応
            m = re.match(r"^\s*\d+\s*[:.]?\s+(.*)$", line)

            if m:
                label = m.group(1).strip()
            else:
                label = line

            labels.append(label)

    return labels


labels = load_labels(label_path)

print("\n認識するクラス:")
for i, label in enumerate(labels):
    print(f"  {i}: {label}")


# ==========================================
# モデル読み込み
# ==========================================
model = safe_load_model(model_path)

print("\nモデルのロード完了")
print("input_shape :", model.input_shape)
print("output_shape:", model.output_shape)


# 入力サイズをモデルから取得
def get_input_size(model):
    shape = model.input_shape

    # Sequential / Functional の通常形式
    if isinstance(shape, tuple):
        h, w = shape[1], shape[2]

        if h is not None and w is not None:
            return int(w), int(h)

    # 万一リスト形式だった場合
    if isinstance(shape, list) and len(shape) == 1:
        shape = shape[0]
        h, w = shape[1], shape[2]

        if h is not None and w is not None:
            return int(w), int(h)

    # Teachable Machine画像モデルの標準サイズ
    print("入力サイズを取得できなかったため224x224を使用します。")
    return 224, 224


INPUT_WIDTH, INPUT_HEIGHT = get_input_size(model)

print(f"推論入力サイズ: {INPUT_WIDTH} x {INPUT_HEIGHT}")


# 出力クラス数の確認
output_shape = model.output_shape

if isinstance(output_shape, tuple):
    num_outputs = output_shape[-1]
elif isinstance(output_shape, list) and len(output_shape) == 1:
    num_outputs = output_shape[0][-1]
else:
    num_outputs = None

if num_outputs is not None and len(labels) != int(num_outputs):
    print(
        f"\n⚠ 注意: モデルの出力クラス数={num_outputs}、"
        f"labels.txt={len(labels)}"
    )
    print("labels.txtがモデルと対応しているか確認してください。")


TensorFlow : 2.21.0
tf_keras  : 2.21.0
Gradio    : 6.29.0
Teachable Machineでエクスポートしたモデル（.h5）をアップロードしてください。


Saving keras_model.h5 to keras_model (1).h5

ラベルファイル（.txt）をアップロードしてください。


Saving labels.txt to labels (1).txt

認識するクラス:
  0: Class 1
  1: Class 2
✓ tf_kerasでモデルを読み込みました

モデルのロード完了
input_shape : (None, 224, 224, 3)
output_shape: (None, 2)
推論入力サイズ: 224 x 224


## 画像認識の実行

次のセルを実行すると Gradio の画面が表示されます。

- **リアルタイム認識**タブ：カメラ欄で録画を開始すると、約0.5秒ごとに認識します。停止ボタンで止まります。
- **画像で認識**タブ：画像のアップロードまたはカメラ撮影で1枚ずつ認識します。

カメラは **Gradio の画面を開いているブラウザ（端末）** のものが使われます。表示された公開URL（`https://～.gradio.live`）をスマホで開けば、スマホのカメラで認識できます。
カメラへのアクセス許可を求められたら許可してください。


In [2]:
# ==========================================
# 画像認識
# ==========================================
def predict_image(image):
    """
    RGB numpy画像を受け取り、
    [認識画像, 認識結果テキスト]を返す。
    """
    if image is None:
        return None, "画像がありません"

    try:
        # PILなどが渡された場合にも対応
        if isinstance(image, Image.Image):
            image = np.array(image.convert("RGB"))

        image = np.asarray(image)

        # RGBA → RGB
        if image.ndim == 3 and image.shape[-1] == 4:
            image = image[:, :, :3]

        # 元画像
        original_image = image.copy()

        # モデルの入力サイズへ変更
        img = cv2.resize(
            image,
            (INPUT_WIDTH, INPUT_HEIGHT),
            interpolation=cv2.INTER_AREA
        )

        # Teachable Machineの標準的な前処理
        img = img.astype(np.float32) / 255.0
        img = np.expand_dims(img, axis=0)

        # 推論
        prediction = model.predict(img, verbose=0)

        # 通常の画像分類モデルは [1, num_classes]
        prediction = np.asarray(prediction)

        if prediction.ndim != 2 or prediction.shape[0] != 1:
            raise ValueError(
                f"想定外のモデル出力shapeです: {prediction.shape}"
            )

        scores = prediction[0]

        max_idx = int(np.argmax(scores))
        confidence = float(scores[max_idx])

        if max_idx >= len(labels):
            class_name = f"class_{max_idx}"
        else:
            class_name = labels[max_idx]

        # 結果テキスト
        result_text = (
            f"予測クラス: {class_name}\n"
            f"確信度: {confidence:.2f}"
        )

        for i, score in enumerate(scores):
            label = labels[i] if i < len(labels) else f"class_{i}"
            result_text += f"\n{label}: {float(score):.2f}"

        # 表示用画像
        result_image = original_image.copy()

        # 大きすぎる場合は表示用だけ縮小
        if result_image.shape[1] > 800:
            scale = 800 / result_image.shape[1]
            new_height = int(result_image.shape[0] * scale)

            result_image = cv2.resize(
                result_image,
                (800, new_height),
                interpolation=cv2.INTER_AREA
            )

        # OpenCVはRGB/BGRどちらでも文字描画自体は問題ない
        cv2.putText(
            result_image,
            f"{class_name}: {confidence:.2f}",
            (10, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )

        return result_image, result_text

    except Exception as e:
        return image, f"認識エラー: {type(e).__name__}: {e}"


# ==========================================
# Gradioインターフェース
# ==========================================
def setup_gradio_interface():

    with gr.Blocks(title="Teachable Machine 画像認識") as interface:

        gr.Markdown("# Teachable Machine 画像認識")
        gr.Markdown(
            f"**モデル:** `{model_path}`  \n"
            f"**ラベル:** `{label_path}`  \n"
            f"**入力サイズ:** {INPUT_WIDTH} × {INPUT_HEIGHT}"
        )

        # ---- リアルタイム認識（ブラウザのWebカメラをストリーミング） ----
        with gr.Tab("リアルタイム認識"):
            with gr.Row():
                with gr.Column():
                    cam = gr.Image(
                        label="カメラ",
                        sources=["webcam"],
                        streaming=True,
                        type="numpy"
                    )
                with gr.Column():
                    cam_output = gr.Image(label="認識結果", type="numpy")
                    cam_text = gr.Textbox(label="検出されたクラス", lines=6)

            # 約0.5秒ごとにブラウザからフレームが送られ、認識される
            cam.stream(
                fn=predict_image,
                inputs=[cam],
                outputs=[cam_output, cam_text],
                stream_every=0.5
            )

        # ---- 画像1枚で認識（アップロード／撮影） ----
        with gr.Tab("画像で認識"):
            with gr.Row():
                with gr.Column():
                    image_input = gr.Image(
                        label="画像入力",
                        sources=["upload", "webcam"],
                        type="numpy"
                    )
                with gr.Column():
                    image_output = gr.Image(label="認識結果", type="numpy")
                    text_output = gr.Textbox(label="検出されたクラス", lines=6)

            image_input.change(
                fn=predict_image,
                inputs=[image_input],
                outputs=[image_output, text_output]
            )

    return interface


# ==========================================
# 起動
# ==========================================
print("Gradioインターフェースを準備しています...")

interface = setup_gradio_interface()

interface.launch(
    share=True,
    debug=True
)


Gradioインターフェースを準備しています...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://eb5af566214d85f508.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://eb5af566214d85f508.gradio.live


カメラ欄の中央にある「Click to Access Webcam」をクリックしてみてください（無反応なときも多い。その場合は次の説明欄へ）。これが許可のきっかけになります。

「Click to Access Webcam」をクリックすると、Edge のアドレスバー付近に「カメラの使用を許可しますか」というポップアップが出るので「許可」を押します。
カメラ欄に映像が映り、その下に開始ボタン（● または「Record」）が出るので押します。
約0.5秒ごとに右側の「認識結果」と「検出されたクラス」が更新されます。



---



ポップアップが出ない、または映像が映らない場合は、Colab のセル内に埋め込まれた画面がカメラを使えていない可能性があります。その場合は公開URL（https://**************.gradio.live）を別タブで開いて上記と同じ操作をしてください。